In [0]:
# What it is doing
# check 1 : Is thetable populated (Workspace..bronze.marc)
# check 2 : Are key columns null ?
# check 3 : Are there duplicate keys (chekc for MATNR + WERKS, if tehr eis duplicate plant, week combination)
# check 4 : 

from pyspark.sql import functions as F
from functools import reduce


# ============================================================
# BRONZE DATA QUALITY GATE
# ============================================================

print("=" * 70)
print("BRONZE DATA QUALITY CHECK")
print("=" * 70)


# ============================================================
# 1. TABLES AND BUSINESS KEYS
# ============================================================

dq_keys = {
    "mara": ["MATNR"],
    "marc": ["MATNR", "WERKS"],
    "mard": ["MATNR", "WERKS", "LGORT"],
    "t001w": ["WERKS"],
    "lfa1": ["LIFNR"],
    "vbak": ["VBELN"],
    "vbap": ["VBELN", "POSNR"],
    "ekko": ["EBELN"],
    "ekpo": ["EBELN", "EBELP"],
    "eket": ["EBELN", "EBELP", "ETENR"],
    "shipment_header": ["shipment_id"],
    "tracking_event": ["shipment_id", "event_type", "event_date"],
    "carrier_master": ["carrier_id"],
    "wms_inventory": ["warehouse_id", "bin_id", "material_id"],
    "pick_task": ["pick_task_id"]
}


# ============================================================
# 2. RUN DATA QUALITY CHECKS
# ============================================================

results = []


for table_name, key_columns in dq_keys.items():

    print("\n" + "-" * 70)
    print(f"CHECKING: {table_name}")
    print("-" * 70)

    try:

        # -----------------------------------------
        # Read Bronze table
        # -----------------------------------------

        df = spark.table(
            f"workspace.bronze.{table_name}"
        )

        row_count = df.count()


        # -----------------------------------------
        # CHECK 1: TABLE NOT EMPTY
        # -----------------------------------------

        row_check = row_count > 0


        # -----------------------------------------
        # CHECK 2: NULL BUSINESS KEYS
        # -----------------------------------------

        null_condition = reduce(
            lambda x, y: x | y,
            [
                F.col(column).isNull()
                for column in key_columns
            ]
        )

        null_key_count = (
            df
            .filter(null_condition)
            .count()
        )

        null_check = null_key_count == 0


        # -----------------------------------------
        # CHECK 3: DUPLICATE BUSINESS KEYS
        # -----------------------------------------

        duplicate_count = (
            df
            .groupBy(*key_columns)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        duplicate_check = duplicate_count == 0


        # -----------------------------------------
        # OVERALL TABLE STATUS
        # -----------------------------------------

        if (
            row_check
            and null_check
            and duplicate_check
        ):
            status = "PASS"
        else:
            status = "FAIL"


        results.append(
            (
                table_name,
                row_count,
                null_key_count,
                duplicate_count,
                status
            )
        )


        # -----------------------------------------
        # PRINT RESULT
        # -----------------------------------------

        print(f"Rows               : {row_count}")
        print(f"Null key records   : {null_key_count}")
        print(f"Duplicate key sets : {duplicate_count}")
        print(f"STATUS             : {status}")


    except Exception as e:

        print(f"ERROR: {str(e)}")

        results.append(
            (
                table_name,
                0,
                -1,
                -1,
                "FAIL"
            )
        )


# ============================================================
# 3. CREATE DQ RESULT TABLE
# ============================================================

dq_results = spark.createDataFrame(
    results,
    [
        "table_name",
        "row_count",
        "null_key_count",
        "duplicate_key_count",
        "status"
    ]
)


# ============================================================
# 4. DISPLAY DQ RESULTS
# ============================================================

display(
    dq_results.orderBy("status", "table_name")
)


# ============================================================
# 5. OVERALL SUMMARY
# ============================================================

total_tables = dq_results.count()

passed_tables = (
    dq_results
    .filter(F.col("status") == "PASS")
    .count()
)

failed_tables = (
    dq_results
    .filter(F.col("status") == "FAIL")
    .count()
)


print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

print(f"Total tables : {total_tables}")
print(f"Passed       : {passed_tables}")
print(f"Failed       : {failed_tables}")


# ============================================================
# 6. WORKFLOW GATE
# ============================================================

if failed_tables > 0:

    print("\n" + "=" * 70)
    print("❌ DATA QUALITY FAILED")
    print("WORKFLOW WILL STOP")
    print("=" * 70)

    failed_table_names = [
        row["table_name"]
        for row in (
            dq_results
            .filter(F.col("status") == "FAIL")
            .select("table_name")
            .collect()
        )
    ]

    print(
        "Failed tables:",
        failed_table_names
    )

    raise Exception(
        f"Bronze Data Quality FAILED. "
        f"{failed_tables} table(s) failed validation."
    )


else:

    print("\n" + "=" * 70)
    print("✅ DATA QUALITY PASSED")
    print("WORKFLOW CAN CONTINUE TO SILVER")
    print("=" * 70)